# Autoencoders in SeqPredNN

By this point, you should be aware of the research problem (have a look at this [document](https://app.eraser.io/workspace/9celyqYK57U4tXAzgLt2?origin=share&elements=twFPvVLW42hEp0iEezmpfQ) if you are not). But to sum up the research, SeqPredNN is a model that predicts the amino acid sequence given a protein's structure. What was found in the release of the model is that it manages to find a different sequence from the original amino acid sequence associated with the protein structure, and when you put this predicted sequence into something like [AlphaFold](https://www.nature.com/articles/s41586-021-03819-2), you get a protein structure that is very similar to the original structure. So the model seems to swap out amino acids and still predict the protein's structure. Now we essentially want to find the range of local structures that can be supported by each of the 20 amino acid residues.

One of the ways that could help us solve this research question is by making use of autoencoders. So this notebook will dive deep into autoencoders, and we will build one from scratch, and hopefully, by the end, you can see how useful they can be.

#### Resources
- An amazing [video](https://www.youtube.com/watch?v=3jmcHZq3A5s) explaining autoencoders

## Autoencoders

Autoencoders are a type of neural networks designed to learn efficient representations of data, in this notebook we will be using them for dimensionality reduction. But essentially the core idea is to encode the input data into a lower-dimensional space and then attempt to reconstruct the original input from this compressed representation (you will see why that will be useful for the research soon).

<img src="media/auto_encoder/autoencoder_network.png" alt="Amino Acid Structure" style="width: 300px;"/>

#### Encoder
The encoder part of the autoencoder transforms the original input into a lower-dimensional representation. The goal is to map the data from its original high-dimensional space into a lower-dimensional space.

#### Embedding Vector
This lower dimension is often referred to as the embedding vector. So we want this vector to capture the essential features of the original input and reduce complexity whilst preserving essential values of the input. 

$$ Embedding Vector = Encoder(Original Features) $$ 

#### Decoder
The decoder then tries to reconstruct the original input from this lower-dimensional embedding vector. This is basically the reverse of the encoding process:

$$ Reconstructed Features = Decoder(Embedding Vector) $$ 

#### Reconstruction Error
The performance of an autoencoder is typically evaluated using the reconstruction error, which is the difference between the original input features and the reconstructed output featuresand it tells us how well the model can recreate the orginal input vector from the lower dimensianl embedding vector.

$$ Reconstruction Error = Original Features - Reconstructed Features $$


#### Importance of Information Loss
What's interesting here is that it tries to reconstruct a higher-dimensional input from a lower-dimensional representation, so we are essneitally asking the decoder to recreate the original input with less information. By forcing the decoder to work with less information, we train the neural network to minimize the reconstruction error, forcing the encoder and decoder to work together in finding the most efficient way to compress the input data into a lower dimension.

If there were no loss of information (if the lower-dimensional space was the same size as the original), the network might just learn to copy the input exactly, which would be trivial and not useful. 


#### Denoising Autoencoders
One of the ways to enforce this information loss is by using Denoising Autoencoders where noise is added to the input before passing it through the network. The network is then trained to reconstruct the original, noise-free input.

$$ Error = Decoder(Encoder(X + noise)) - X $$

### Applications of Autoencoders in SeqPredNN

We still aim to answer **"What common local protein structural motifs are supported by specific amino acids?"**

To do this, we'll build an autoencoder model that takes in feature vectors describing the local coordinate system of amino acids and their 16 nearest neighbors. The model will compress these feature vectors into a lower-dimensional latent space, enabling us to cluster amino acid features and identify local structures supported by different amino acids.

We will be focusing heavily on the latent space representation of the features. After training the model, we will remove the decoder to concentrate purely on the latent space (lower-dimensional) representation of the features. Visualizing this space can reveal clusters of similar structures, which can further be refined with clustering algorithms.

By clustering in this lower dimension, as opposed to clustering in the higher dimension of our original feature vector, we are clustering in a space where the model understands the underlying structural features. This latent space is a representation of how the model interprets amino acid features.

<img src="media/auto_encoder/autoencoder_model.png" alt="Amino Acid Structure"/>


### Workflow Example: Analyzing Local Structural Motifs Using Autoencoders
To show you all of this in a more practical sence, let us run through an exmaple of a workflow using the autoendocing model and how it will help us answer our reseach question. 

**1. Featurization:**
   - **Input:** Amino acid sequence (e.g., MKTEIAGC)
   - **Process:** Generate 8 feature vectors (180D each) using SeqPredNN.
   
**2. Encoding to Latent Space:**
   - **Input:** 8 feature vectors (180D each)
   - **Process:** Encode to 3D latent space vectors.
   
**3. Visualization:**
   - **Plot:** 3D latent space vectors.
   - **Observe:** Clustering patterns and distances.
   
**4. Clustering:**
   - **Process:** Apply a clustering algorithm to latent space vectors.
   - **Example Clusters:** 
     - Cluster 1: M, I
     - Cluster 2: K, E, G
     - Cluster 3: T, A, C

**5. Analysis:**
   - **Structural Motifs:** Clusters reveal possible local motifs.
   - **Examples:**
     - Cluster 1: Hydrophobic residues
     - Cluster 2: Charged residues
     - Cluster 3: Structured region preferences